# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695

## Clustering

**Flujo de trabajo:**
1. Cargar objeto integrado del Paso anterior
2. Ejecutar Leiden con varias resoluciones
3. Visualizar clusters en UMAP post-Harmony
4. Estadísticas básicas de cada cluster
5. Guardar objeto con clusters

# · Importaciones y configuración

In [ ]:
# Montar Google Drive y prepara el gestor de entornos Conda
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:14
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda del proyecto (environment.yml), con las
# versiones exactas de todas las dependencias fijadas para reproducibilidad.

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment.yml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

from pathlib import Path
import leidenalg
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import scanpy as sc
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.preprocessing import LabelEncoder
import yaml

sc.settings.verbosity = 1

# · Configuración de rutas

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

INPUT_PATH  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['04_integrated']}/{PARAMS['outputs']['integrated_h5ad']}"
OUTPUT_DIR  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['05_clustered']}"
OUTPUT_PATH = f"{OUTPUT_DIR}/{PARAMS['outputs']['clustered_h5ad']}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TABLES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables']['05_clustering']}"
Path(TABLES_DIR).mkdir(parents=True, exist_ok=True)

FIGURES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['figures']['05_clustering']}"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
sc.settings.figdir = FIGURES_DIR

BATCH_KEY     = "sample_id"
CONDITION_KEY = "condition"
N_PCS         = 20

# Resoluciones a probar. Se ejecutan todas y se guardan en .obs.
# Elegimos la que mejor separe los tipos celulares.
RESOLUTIONS = [0.3, 0.5, 0.8, 1.0]

PALETTE = {'HC': '#2E86AB', 'UC': '#E84855', 'CD': '#F4A261'}

In [ ]:
adata = sc.read_h5ad(INPUT_PATH)
print(f"   {adata.n_obs:,} células × {adata.n_vars:,} genes")

# Normalizar tipos de índice
adata.obs.index = adata.obs.index.astype(str); adata.var.index = adata.var.index.astype(str)
if adata.raw is not None: adata.raw.var.index = adata.raw.var.index.astype(str)

# Verificar que tenemos lo necesario del paso anterior
assert 'X_pca_harmony' in adata.obsm, \
    "ERROR: X_pca_harmony no encontrado."
assert 'X_umap_harmony' in adata.obsm, \
    "ERROR: X_umap_harmony no encontrado."

# Verificar o recalcular vecindario sobre Harmony
if 'neighbors_harmony' not in adata.uns:
    print("   neighbors_harmony no encontrado — recalculando...")
    sc.pp.neighbors(
        adata,
        n_pcs=N_PCS,
        use_rep='X_pca_harmony',
        n_neighbors=15,
        key_added='neighbors_harmony'
    )
    print("   Vecindario recalculado")
else:
    print("   neighbors_harmony encontrado")

# Usar el UMAP de Harmony para todas las visualizaciones
umap = adata.obsm['X_umap_harmony']

   43,388 células × 33,538 genes
   neighbors_harmony encontrado


# 🔬 Clustering Leiden con múltiples resoluciones

Se ejecuta con varias resoluciones para poder comparar cuántos clusters produce cada una y elegir la más biológicamente informativa.


In [ ]:
for res in RESOLUTIONS:
    key = f"leiden_r{res}"
    sc.tl.leiden(
        adata,
        resolution=res,
        neighbors_key='neighbors_harmony',
        key_added=key,
        random_state=42,
        flavor="leidenalg"
    )
    n_clusters = adata.obs[key].nunique()
    print(f"   Resolución {res}: {n_clusters} clusters → guardado en obs['{key}']")


   Resolución 0.3: 16 clusters → guardado en obs['leiden_r0.3']
   Resolución 0.5: 17 clusters → guardado en obs['leiden_r0.5']
   Resolución 0.8: 21 clusters → guardado en obs['leiden_r0.8']
   Resolución 1.0: 24 clusters → guardado en obs['leiden_r1.0']


Antes de fijar la resolución de trabajo, se contrasta cuantitativamente frente a la
resolución inmediatamente inferior.

In [ ]:
def calc_metrics(res_key):
    enc = LabelEncoder().fit_transform(adata.obs[res_key].values)
    sil = silhouette_score(adata.obsm['X_pca_harmony'][:, :N_PCS], enc,
                            sample_size=5000, random_state=42)
    db  = davies_bouldin_score(adata.obsm['X_pca_harmony'][:, :N_PCS], enc)
    return sil, db

sil_r05, db_r05 = calc_metrics('leiden_r0.5')
sil_r08, db_r08 = calc_metrics('leiden_r0.8')

print(f"   Silhouette  r0.5={sil_r05:.4f}  vs  r0.8={sil_r08:.4f}")
print(f"   Davies-Bouldin r0.5={db_r05:.4f}  vs  r0.8={db_r08:.4f}")


   Silhouette  r0.5=0.3422  vs  r0.8=0.2629
   Davies-Bouldin r0.5=0.9109  vs  r0.8=1.1483


In [ ]:
MAIN_RESOLUTION = 0.8
MAIN_KEY        = f"leiden_r{MAIN_RESOLUTION}"
print(f"\n   Resolución principal seleccionada: {MAIN_RESOLUTION} ({MAIN_KEY})")
print(f"   ({adata.obs[MAIN_KEY].nunique()} clusters)")


   Resolución principal seleccionada: 0.8 (leiden_r0.8)
   (21 clusters)


# · Visualización de clusters en UMAP

In [ ]:
# FIGURA 1: UMAP combinado con etiquetas de cluster
# Todas las células en el mismo gráfico, coloreadas por cluster,
# con etiquetas numéricas para facilitar la identificación visual.

labels      = adata.obs[MAIN_KEY].values
cluster_ids = sorted(adata.obs[MAIN_KEY].unique(), key=int)
n_clusters = adata.obs[MAIN_KEY].nunique()
cmap_cl     = plt.get_cmap('tab20', n_clusters)

fig, ax = plt.subplots(figsize=(12, 10))
fig.suptitle(
    f"UMAP post-Harmony — Clustering Leiden (resolución {MAIN_RESOLUTION})\n"
    f"{n_clusters} clusters | {adata.n_obs:,} células",
    fontsize=13, fontweight='bold'
)

# Dibujar cada cluster con su color
for i, cl in enumerate(cluster_ids):
    mask = labels == cl
    ax.scatter(umap[mask, 0], umap[mask, 1],
               s=0.5, alpha=0.5, color=cmap_cl(i), rasterized=True)

# Añadir etiqueta numérica en el centroide de cada cluster
for i, cl in enumerate(cluster_ids):
    mask   = labels == cl
    cx, cy = umap[mask, 0].mean(), umap[mask, 1].mean()
    ax.text(cx, cy, str(cl), fontsize=8, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor='gray', alpha=0.7))

ax.set_xlabel("UMAP 1", fontsize=11)
ax.set_ylabel("UMAP 2", fontsize=11)
ax.set_title(f"Resolución {MAIN_RESOLUTION} → {n_clusters} clusters", fontsize=10)

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/umap_clusters_labeled_{MAIN_KEY}.png",
            dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: umap_clusters_labeled_{MAIN_KEY}.png")

# ── FIGURA 2: UMAP dividido por muestra (split by sample) ────────────────────
# Verificar que ningún cluster aparece solo en una muestra.

sample_list = sorted(adata.obs[BATCH_KEY].unique())
n_samples   = len(sample_list)
ncols = 6
nrows = int(np.ceil(n_samples / ncols))

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 3.5, nrows * 3.5))
fig.suptitle(
    f"UMAP dividido por muestra — verificación de artefactos técnicos\n",
    fontsize=12, fontweight='bold'
)

axes_flat = axes.flatten()

for idx, sample in enumerate(sample_list):
    ax = axes_flat[idx]
    # Dibujar todas las células en gris de fondo
    ax.scatter(umap[:, 0], umap[:, 1], s=0.1, alpha=0.1,
               color='lightgray', rasterized=True)
    # Superponer las células de esta muestra coloreadas por cluster
    mask_s = adata.obs[BATCH_KEY] == sample
    for i, cl in enumerate(cluster_ids):
        mask_cl = (labels == cl) & mask_s
        if mask_cl.sum() > 0:
            ax.scatter(umap[mask_cl, 0], umap[mask_cl, 1],
                       s=0.3, alpha=0.6, color=cmap_cl(i), rasterized=True)
    # Obtener condición de la muestra para el título
    cond = adata.obs[CONDITION_KEY][mask_s].iloc[0]
    n_cells_s = mask_s.sum()
    ax.set_title(f"{sample.split('_')[-1]}\n({cond}, {n_cells_s:,} céls)",
                 fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

# Ocultar subplots vacíos si n_samples no llena la cuadrícula
for idx in range(n_samples, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/umap_split_by_sample_{MAIN_KEY}.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: umap_split_by_sample_{MAIN_KEY}.png")

# ── FIGURA 3: Comparativa de resoluciones ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle(
    "Clustering Leiden — comparativa de resoluciones\n",
    fontsize=12, fontweight='bold'
)
for ax, res in zip(axes.flatten(), RESOLUTIONS):
    key_r  = f"leiden_r{res}"
    labs_r = adata.obs[key_r].values
    n_cl_r = adata.obs[key_r].nunique()
    cmap_r = plt.get_cmap('tab20', n_cl_r)
    for i, cl in enumerate(sorted(adata.obs[key_r].unique(), key=int)):
        mask = labs_r == cl
        ax.scatter(umap[mask, 0], umap[mask, 1],
                   s=0.4, alpha=0.4, color=cmap_r(i), rasterized=True)
    ax.set_title(f"Resolución {res} → {n_cl_r} clusters", fontsize=10)
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/clustering_resolution_comparison.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: clustering_resolution_comparison.png")

# ── FIGURA 4: UMAP coloreado por condición ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for cond in ['HC', 'UC', 'CD']:
    mask = adata.obs[CONDITION_KEY] == cond
    ax.scatter(umap[mask, 0], umap[mask, 1],
               s=0.5, alpha=0.3, color=PALETTE[cond], label=cond, rasterized=True)
ax.set_title("UMAP post-Harmony — Condición biológica (HC/UC/CD)", fontsize=11)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.legend(markerscale=8, fontsize=11, frameon=False)
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/umap_by_condition_{MAIN_KEY}.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: umap_by_condition_{MAIN_KEY}.png")


   → Guardada: umap_clusters_labeled_{MAIN_KEY}.png
   → Guardada: umap_split_by_sample_{MAIN_KEY}.png
   → Guardada: clustering_resolution_comparison.png
   → Guardada: umap_by_condition_{MAIN_KEY}.png


# · Estadísticas básicas de cada cluster
Para cada cluster se calcula:
- Número de células
- Mediana de genes y UMIs
- Mediana de % mito
- Distribución por condición

In [ ]:
# Recalcular métricas QC si no están
if 'n_genes_by_counts' not in adata.obs.columns:
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
    )

stats_list = []
for cl in sorted(adata.obs[MAIN_KEY].unique(), key=int):
    mask  = adata.obs[MAIN_KEY] == cl
    sub   = adata.obs[mask]
    n_cells = mask.sum()

    # Distribución por condición
    cond_counts = sub[CONDITION_KEY].value_counts()
    pct_hc = 100 * cond_counts.get('HC', 0) / n_cells
    pct_uc = 100 * cond_counts.get('UC', 0) / n_cells
    pct_cd = 100 * cond_counts.get('CD', 0) / n_cells

    stats_list.append({
        'cluster':      cl,
        'n_cells':      n_cells,
        'med_genes':    sub['n_genes_by_counts'].median(),
        'med_umis':     sub['total_counts'].median(),
        'med_mito_pct': sub['pct_counts_mt'].median(),
        'pct_HC':       round(pct_hc, 1),
        'pct_UC':       round(pct_uc, 1),
        'pct_CD':       round(pct_cd, 1),
    })

stats_df = pd.DataFrame(stats_list)
print(stats_df.to_string(index=False, float_format='%.1f'))

# Guardar CSV
stats_path = f"{TABLES_DIR}/cluster_stats_r{MAIN_RESOLUTION}.csv"
stats_df.to_csv(stats_path, index=False, float_format='%.1f')
print(f"\n   → CSV guardado: {stats_path}")

# Alertas para clusters potencialmente problemáticos
high_mito_clusters = stats_df[stats_df['med_mito_pct'] > 30]['cluster'].tolist()
low_gene_clusters  = stats_df[stats_df['med_genes'] < 300]['cluster'].tolist()

if high_mito_clusters:
    print(f"\n     Clusters con mediana mito > 30%: {high_mito_clusters}")
if low_gene_clusters:
    print(f"     Clusters con mediana genes < 300: {low_gene_clusters}")

# ── Figura 5: composición de condición por cluster ────────────────────────────
# Un cluster dominado por UC (>70%) puede representar un tipo celular
# específico de la inflamación ulcerosa.
n_cl = len(stats_df)
fig, ax = plt.subplots(figsize=(max(10, n_cl), 5))
x       = range(len(stats_df))
xlabels = [f"C{r['cluster']}\n({int(r['n_cells'])})" for _, r in stats_df.iterrows()]

ax.bar(x, stats_df['pct_HC'], label='HC', color=PALETTE['HC'], alpha=0.85)
ax.bar(x, stats_df['pct_UC'], bottom=stats_df['pct_HC'],
       label='UC', color=PALETTE['UC'], alpha=0.85)
ax.bar(x, stats_df['pct_CD'],
       bottom=stats_df['pct_HC'] + stats_df['pct_UC'],
       label='CD', color=PALETTE['CD'], alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=8)
ax.set_ylabel("% células por condición")
ax.set_title(
    f"Composición por condición en cada cluster (resolución {MAIN_RESOLUTION})\n",
    fontsize=10
)
ax.axhline(50, color='black', lw=0.8, ls='--', alpha=0.5)
ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/cluster_condition_composition.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: cluster_condition_composition.png")

cluster  n_cells  med_genes  med_umis  med_mito_pct  pct_HC  pct_UC  pct_CD
      0     4865     1207.0    3303.0          10.3    36.3    29.7    34.0
      1     4673      774.0    1849.0          18.0    51.5     5.2    43.3
      2     4436     1248.5    3715.0           8.3    23.3    35.8    40.9
      3     4162     1513.0   17976.5           3.4    30.3    45.5    24.1
      4     3675     1284.0   20055.0           2.5    27.7    50.6    21.6
      5     3451      920.0    2739.0           9.5    19.8    39.2    41.0
      6     2939     1394.0   17686.0           2.3     5.6    70.8    23.6
      7     2478     1501.0    4067.0          11.7    47.9    24.3    27.8
      8     2421      963.0    3353.0          29.7    58.5    10.5    31.1
      9     2226     1240.5    4047.5          10.6    16.2    45.7    38.1
     10     2187      864.0    2828.0          24.2    54.6     6.5    38.9
     11     2132     1279.5    3824.0           8.7    14.1    35.4    50.5
     12     

# · Evaluación de calidad del clustering


In [ ]:
le        = LabelEncoder()
clust_enc = le.fit_transform(adata.obs[MAIN_KEY].values)

# Reutiliza los valores ya calculados en la comparación de resoluciones
sil_global, db_index = sil_r08, db_r08
print(f"   Silhouette global  : {sil_global:.4f}  "
      f"(0.2-0.5 = razonable en scRNA-seq)")
print(f"   Davies-Bouldin idx : {db_index:.4f}  ")

# Silhouette por cluster
sil_samples = silhouette_samples(
    adata.obsm['X_pca_harmony'][:, :N_PCS], clust_enc
)
sil_per_cluster = (
    pd.DataFrame({'cluster': adata.obs[MAIN_KEY].values,
                  'silhouette': sil_samples})
    .groupby('cluster')['silhouette'].mean()
    .sort_values()
)

sil_per_cluster.to_csv(f"{TABLES_DIR}/silhouette_by_cluster_{MAIN_KEY}.csv", header=['silhouette'])

# Guardar resumen de métricas
summary_df = pd.DataFrame({
    'Métrica':      ['Silhouette global', 'Davies-Bouldin index', 'N clusters'],
    'Valor':        [round(sil_global, 4), round(db_index, 4), n_clusters],
    'Interpretación': [
        '0.2-0.5 = razonable en scRNA-seq',
        'Más bajo = mejor',
        f'Resolución {MAIN_RESOLUTION}'
    ]
})
summary_df.to_csv(f"{TABLES_DIR}/cluster_quality_metrics_{MAIN_KEY}.csv", index=False)

print(f"\n   Resumen guardado en: cluster_quality_metrics_{MAIN_KEY}.csv")

# ── FIGURA 6: Silhouette por cluster ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, max(5, n_clusters * 0.4)))
colors_sil = ['#E84855' if v < 0 else '#2E86AB' for v in sil_per_cluster.values]
sil_per_cluster.plot(kind='barh', ax=ax, color=colors_sil, edgecolor='none')
ax.axvline(x=0,   color='black', lw=1.0, ls='-')
ax.axvline(x=0.2, color='orange', lw=1.2, ls='--', alpha=0.8,
           label='Mínimo aceptable (0.2)')
ax.axvline(x=0.5, color='green',  lw=1.2, ls='--', alpha=0.8,
           label='Bueno (0.5)')
ax.set_xlabel('Silhouette score promedio', fontsize=10)
ax.set_ylabel('Cluster', fontsize=10)
ax.set_title(
    f'Silhouette score por cluster ({MAIN_KEY})\n',
    fontsize=10
)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/silhouette_by_cluster_{MAIN_KEY}.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: silhouette_by_cluster_{MAIN_KEY}.png")

# ── FIGURA 7: Heatmap condición × cluster ────────────────────────────────────
cond_cross = pd.crosstab(
    adata.obs[MAIN_KEY], adata.obs[CONDITION_KEY], normalize='index'
)
fig, ax = plt.subplots(figsize=(6, max(6, n_clusters * 0.45)))
sns.heatmap(
    cond_cross,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    ax=ax,
    vmin=0, vmax=0.8,
    linewidths=0.3, linecolor='white'
)
ax.set_title(
    f'Proporción de condiciones por cluster ({MAIN_KEY})\n',
    fontsize=10
)
ax.set_ylabel('Cluster'); ax.set_xlabel('Condición')
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/cluster_condition_heatmap_{MAIN_KEY}.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: cluster_condition_heatmap_{MAIN_KEY}.png")

# ── FIGURA 8: Heatmap muestra × cluster ──────────────────────────────────────
# Normalizado por fila (por cluster) para comparar proporciones.
sample_cross = pd.crosstab(
    adata.obs[MAIN_KEY], adata.obs[BATCH_KEY], normalize='index'
)
fig, ax = plt.subplots(figsize=(14, max(6, n_clusters * 0.45)))
sns.heatmap(
    sample_cross,
    cmap='viridis',
    ax=ax,
    cbar_kws={'label': 'Proporción de células'},
    linewidths=0.2,
    linecolor='white'
)
ax.set_title(
    f'Proporción de células por cluster y muestra ({MAIN_KEY})\n',
    fontsize=10
)
ax.set_xlabel('Muestra'); ax.set_ylabel('Cluster')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/cluster_sample_heatmap_{MAIN_KEY}.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: cluster_sample_heatmap_{MAIN_KEY}.png")


   Silhouette global  : 0.2629  (0.2-0.5 = razonable en scRNA-seq)
   Davies-Bouldin idx : 1.1483  

   Resumen guardado en: cluster_quality_metrics_leiden_r0.8.csv
   → Guardada: silhouette_by_cluster_{MAIN_KEY}.png
   → Guardada: cluster_condition_heatmap_{MAIN_KEY}.png
   → Guardada: cluster_sample_heatmap_{MAIN_KEY}.png


# · Guardar objeto con clusters

In [ ]:
print(f"\n→ Guardando en {OUTPUT_PATH} ...")
adata.write_h5ad(OUTPUT_PATH, compression='gzip')

print(f"  CLUSTERING COMPLETADO")
print(f"  Resolución principal ({MAIN_RESOLUTION}): "
      f"{adata.obs[MAIN_KEY].nunique()} clusters")